# Estimación de Edad Cerebral
## VAE sobre Conectividad Funcional + XGBoost

Este notebook implementa un pipeline de estimación de edad cerebral a partir de dos fuentes de datos de neuroimagen:

- **Conectividad funcional (FC):** matrices de correlación entre regiones cerebrales, procesadas mediante un Autoencoder Variacional (β-VAE) para obtener representaciones latentes compactas.
- **Medidas estructurales (T1w):** volúmenes regionales derivados de VBM sobre el atlas AAL.

El regresor final (XGBoost) integra los embeddings del VAE con las features estructurales para predecir la edad del sujeto. La diferencia entre la edad predicha y la cronológica constituye el *brain-age gap*, un biomarcador asociado a procesos de neurodegeneración.

### Etapas

1. Construcción de la cohorte (intersección FC ∩ Metadata ∩ T1w)
2. Separación holdout estratificada (trainval / test)
3. Optimización de hiperparámetros del VAE mediante Optuna (objetivo: MAE de edad)
4. Entrenamiento del VAE final sobre trainval completo
5. Extracción de embeddings latentes (μ del encoder)
6. Optimización de hiperparámetros de XGBoost con validación cruzada
7. Evaluación final sobre el conjunto de test
8. Ablaciones para cuantificar la contribución de cada fuente de features

### Configuración del entorno

Resolución de la raíz del proyecto y configuración de paths. Se detecta automáticamente si el notebook se ejecuta desde `Code/` o desde `Code/notebooks/`, de modo que `ROOT` siempre apunte al directorio `Code/` (donde reside `src/`).

In [1]:
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / "src").exists():
    ROOT = _cwd
elif (_cwd.parent / "src").exists():
    ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {_cwd}")

REPO = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Code root:", ROOT)
print("Repo root:", REPO)


Code root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Code
Repo root: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci


### Imports, configuración global y rutas

Se importan todos los módulos del pipeline (`src/`), se configuran las rutas a los datos crudos y al directorio de salida, y se fija la semilla global para reproducibilidad. Los parámetros principales del experimento (proporción de test, número de folds, uso de Optuna, etc.) se definen en `ExperimentConfig`.

In [2]:
import logging
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import Paths, ExperimentConfig
from src.utils_seed import set_global_seed
from src.cohort import build_final_cohort_df
from src.splits import make_holdout_split, make_kfold_splits, save_splits, load_splits
from src.data_io import load_fc_vectors_for_ids, vector_to_matrix
from src.vae_train import train_vae_kfold, train_vae_final
from src.vae_optuna_age import folds_ids_to_indices
from src.embeddings import encode_mu, save_embeddings, load_embeddings
from src.xgb_train import build_feats, clean_xy, train_xgb, eval_xgb, build_feature_names
from src.metrics import regression_metrics
from src.figures import (
    plot_vae_history,
    plot_reconstructions,
    plot_pred_scatter,
    plot_feature_importance,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

paths = Paths(
    excel_path=REPO / "Data" / "datos-redlat.xlsx",
    fc_folder=REPO / "Data" / "fc_mats",
    t1w_csv_path=REPO / "Data" / "Redlat_VGM_AAL_.csv",
    out_dir=REPO / "Outputs",
)

cfg = ExperimentConfig(
    seed=42,
    diagnoses_to_use=("CN", "AD", "FTD"),
    test_size=0.10,
    k_folds=5,
    fisher_z=True,
    reuse_artifacts=True,
    use_optuna=True,
    optuna_xgb_trials=100,  # Keep active for potential CN-only experiment
    optuna_vae_trials=0,     # Optimization complete (plateau at MAE≈6.43)
)

set_global_seed(cfg.seed)
paths.out_dir.mkdir(parents=True, exist_ok=True)
print("Output directory:", paths.out_dir.resolve())


2026-02-16 17:34:14.105316: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-16 17:34:14.625957: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 17:34:15.695916: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Output directory: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs


/home/nico/Desktop/5to/TesisFinal/.tesis-final-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Datos

Construcción de la cohorte final (intersección de las tres fuentes de datos), partición holdout estratificada y carga de las matrices de conectividad funcional.

In [3]:
cohort_df = build_final_cohort_df(
    excel_path=paths.excel_path,
    fc_folder=paths.fc_folder,
    t1w_csv_path=paths.t1w_csv_path,
    diagnoses_to_use=cfg.diagnoses_to_use,
)
print("Cohort shape:", cohort_df.shape)
cohort_df.head()


INFO | src.cohort | Found 4 subjects with multiple T1w entries; collapsing by mean.
INFO | src.cohort | Final cohort: (1245, 120) (1245 subjects, diagnoses: {'CN': 526, 'AD': 422, 'FTD': 297})


Cohort shape: (1245, 120)


,record_id,age,sex,diagnosis,t1_0000,t1_0001,t1_0002,t1_0003,t1_0004,t1_0005,...,t1_0106,t1_0107,t1_0108,t1_0109,t1_0110,t1_0111,t1_0112,t1_0113,t1_0114,t1_0115
0,AF025,75.0,Female,CN,0.005376,0.004817,0.005227,0.006189,0.001507,0.001747,...,0.000194,0.000192,0.000039,0.000213,0.000882,0.000687,0.000384,0.000399,0.000295,0.000059
1,AF036,74.0,Female,AD,0.005547,0.005271,0.005518,0.006763,0.001590,0.002115,...,0.000151,0.000188,0.000040,0.000222,0.001098,0.000827,0.000452,0.000482,0.000354,0.000058
2,AF037,71.0,Male,FTD,0.006739,0.005642,0.005663,0.007002,0.001325,0.001742,...,0.000165,0.000175,0.000043,0.000309,0.001203,0.000837,0.000505,0.000580,0.000411,0.000060
3,AF063,73.0,Female,CN,0.005769,0.004920,0.005110,0.006456,0.001601,0.001931,...,0.000187,0.000219,0.000054,0.000288,0.001204,0.000931,0.000562,0.000527,0.000430,0.000071
4,AF065,65.0,Male,CN,0.005791,0.005200,0.005685,0.006578,0.001662,0.001859,...,0.000163,0.000163,0.000055,0.000302,0.001074,0.000845,0.000538,0.000591,0.000474,0.000073


### Partición holdout estratificada

Se cargan (o crean) las particiones de datos: un holdout estratificado por sexo×diagnóstico (90% trainval, 10% test) y 5 folds de validación cruzada sobre trainval.

In [4]:
splits_path = paths.out_dir / "splits" / f"splits_seed{cfg.seed}_test{cfg.test_size}.json"

if cfg.reuse_artifacts and splits_path.exists():
    splits = load_splits(splits_path)
else:
    holdout = make_holdout_split(cohort_df, seed=cfg.seed, test_size=cfg.test_size)
    folds = make_kfold_splits(holdout["trainval_ids"], seed=cfg.seed, k=cfg.k_folds)
    splits = {"holdout": holdout, "folds": folds}
    save_splits(splits_path, splits)

trainval_ids = splits["holdout"]["trainval_ids"]
test_ids = splits["holdout"]["test_ids"]

print("Trainval:", len(trainval_ids), "Test:", len(test_ids))
print("First trainval IDs:", trainval_ids[:5])
print("First test IDs:", test_ids[:5])


Trainval: 1120 Test: 125
First trainval IDs: ['AF025', 'AF036', 'AF037', 'AF063', 'AF075']
First test IDs: ['AF065', 'AF098', 'BE00143', 'BE00171', 'BE00197']


### Carga de vectores FC

Se cargan las matrices de conectividad funcional para trainval y test, vectorizadas (triángulo superior, 6670 features) y con transformación Fisher z aplicada.

In [5]:
X_trainval = load_fc_vectors_for_ids(paths.fc_folder, trainval_ids, apply_fisher_z=cfg.fisher_z)
X_test = load_fc_vectors_for_ids(paths.fc_folder, test_ids, apply_fisher_z=cfg.fisher_z)

print("X_trainval:", X_trainval.shape, "X_test:", X_test.shape)
if cfg.fisher_z:
    print(f"Fisher z range: [{X_trainval.min():.3f}, {X_trainval.max():.3f}]")


INFO | src.data_io | Applied Fisher z-transform to 1120 vectors.
INFO | src.data_io | Applied Fisher z-transform to 125 vectors.


X_trainval: (1120, 6670) X_test: (125, 6670)
Fisher z range: [-1.759, 3.351]


### Hiperparámetros del VAE (valores por defecto)

Se definen los hiperparámetros del β-VAE como fallback. Cuando `use_optuna=True`, estos valores se sobreescriben con los mejores encontrados por la búsqueda de Optuna (celda siguiente). Cuando `use_optuna=False`, se usan directamente estos valores (corresponden a los óptimos encontrados previamente).

In [6]:
vae_hparams = dict(
    hidden_dims=[512],
    latent_dim=224,
    beta_target=0.006220928747773222,
    warmup_ep=81,
    l2_reg=2.897389671945472e-07,
    lr=0.0004477872059719103,
    recon_kind="mae",
    drop_rate=0.03513348441506047,
    activation="elu",
    norm_kind="layernorm",
    max_epochs=300,
    batch_size=64,
    clipnorm=1.0,
)


## Autoencoder Variacional

Optimización de hiperparámetros del β-VAE, entrenamiento por validación cruzada y extracción de embeddings latentes a partir de la conectividad funcional.

In [7]:
from src.vae_optuna_age import run_vae_optuna_for_age, _HIDDEN_DIMS_MAP

# Align auxiliary arrays to trainval_ids
_tmp = cohort_df.set_index("record_id") if "record_id" in cohort_df.columns else cohort_df
t1_cols = [c for c in _tmp.columns if c.startswith("t1_")]
diag_map = {"CN": 0.0, "AD": 1.0, "FTD": 2.0}

y_trainval = _tmp.loc[trainval_ids, "age"].to_numpy(dtype=float)
sex_trainval = (_tmp.loc[trainval_ids, "sex"].astype(str) == "Male").to_numpy(dtype=np.float32)
diag_trainval = _tmp.loc[trainval_ids, "diagnosis"].map(diag_map).to_numpy(dtype=np.float32)
T1_trainval = _tmp.loc[trainval_ids, t1_cols].to_numpy(dtype=np.float32)

if np.any(~np.isfinite(diag_trainval)):
    raise ValueError("Unknown diagnosis labels found. Check diagnoses_to_use and diag_map.")

folds_idx = folds_ids_to_indices(trainval_ids, splits["folds"])

# VAE parameters held constant during Optuna search
vae_fixed = dict(
    l2_reg=2.897389671945472e-07,
    activation="elu",
    norm_kind="layernorm",
    recon_kind="mae",
    clipnorm=1.0,
    batch_size=64,
    max_epochs=80,
    patience=10,
)

# XGB defaults for downstream evaluation (properly tuned in a later stage)
xgb_fixed = {
    "n_estimators": 800,
    "max_depth": 5,
    "learning_rate": 0.02,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 1e-4,
    "reg_lambda": 1.0,
    "min_child_weight": 1.0,
    "gamma": 1.0,
}

optuna_dir = paths.out_dir / "optuna" / f"vae_age_seed{cfg.seed}"
best_json = optuna_dir / "vae_optuna_age_best.json"

# Diagnosis excluded: its association with age confounds brain-age estimation.
# Sex included: documented neuroanatomical effects on brain aging.
use_t1w  = True
use_sex  = True
use_diag = False

if cfg.use_optuna:
    if cfg.reuse_artifacts and best_json.exists():
        with open(best_json, "r", encoding="utf-8") as f:
            best = json.load(f)
    else:
        best = run_vae_optuna_for_age(
            X_trainval_fc=X_trainval,
            y_trainval=y_trainval,
            T1_trainval=T1_trainval,
            sex_trainval=sex_trainval,
            diag_trainval=diag_trainval,
            folds_idx=folds_idx,
            out_dir=optuna_dir,
            seed=cfg.seed,
            n_trials=cfg.optuna_vae_trials,
            vae_fixed=vae_fixed,
            xgb_fixed_params=xgb_fixed,
            use_sex=use_sex,
            use_diag=use_diag,
            use_t1w=use_t1w,
            study_name="vae_pipe_v2",
            storage_path=optuna_dir / "vae_pipe_v2.db",
        )

    print("Best CV MAE:", best["best_value_mae"])
    print("Best params:", best["best_params"])

    bp = best["best_params"]
    vae_hparams = dict(
        hidden_dims=_HIDDEN_DIMS_MAP[str(bp["hidden_dims"])],
        latent_dim=int(bp["latent_dim"]),
        beta_target=float(bp["beta_target"]),
        warmup_ep=int(bp["warmup_ep"]),
        l2_reg=float(vae_fixed["l2_reg"]),
        lr=float(bp["lr"]),
        recon_kind=str(vae_fixed["recon_kind"]),
        drop_rate=float(bp["drop_rate"]),
        activation=str(vae_fixed["activation"]),
        norm_kind=str(vae_fixed["norm_kind"]),
        max_epochs=300,
        batch_size=int(vae_fixed["batch_size"]),
        clipnorm=float(vae_fixed["clipnorm"]),
    )

    print("VAE hparams:", {k: vae_hparams[k] for k in
          ["hidden_dims", "latent_dim", "beta_target", "lr", "drop_rate", "warmup_ep"]})
else:
    assert "vae_hparams" in dir(), "vae_hparams must be defined when use_optuna=False."
    print("Using manually defined vae_hparams.")


Best CV MAE: 6.432324881213051
Best params: {'latent_dim': 64, 'hidden_dims': '512', 'beta_target': 0.056663247229966504, 'lr': 0.001892443497356961, 'drop_rate': 0.036861053246000725, 'warmup_ep': 73, 'embedding_kind': 'z'}
VAE hparams: {'hidden_dims': [512], 'latent_dim': 64, 'beta_target': 0.056663247229966504, 'lr': 0.001892443497356961, 'drop_rate': 0.036861053246000725, 'warmup_ep': 73}


### Entrenamiento K-Fold y modelo final del VAE

Se entrena el β-VAE en dos fases: primero K-Fold (5 folds con EarlyStopping) para determinar el número óptimo de épocas, y luego un entrenamiento final sobre todo trainval con ese número de épocas.

In [8]:
from src.vae_train import load_vae_from_dir, load_history

vae_dir = paths.out_dir / "vae"
kfold_dir = vae_dir / f"kfold_k{cfg.k_folds}_seed{cfg.seed}"
final_vae_dir = vae_dir / ("vae_final_trainval_optuna" if cfg.use_optuna else "vae_final_trainval")

if not (cfg.reuse_artifacts and (kfold_dir / "kfold_summary.json").exists()):
    ksum = train_vae_kfold(
        X_trainval,
        out_dir=kfold_dir,
        seed=cfg.seed,
        k=cfg.k_folds,
        fold_indices=folds_idx,
        **vae_hparams,
    )
else:
    with open(kfold_dir / "kfold_summary.json", "r", encoding="utf-8") as f:
        ksum = json.load(f)

print("KFold mean val recon loss:", ksum["mean_best_val_recon_loss"])
print("Suggested epochs for final:", ksum["suggested_epochs_for_final"])

final_weights = final_vae_dir / "vae.weights.h5"
final_hparams = final_vae_dir / "hparams.json"
final_history_json = final_vae_dir / "history.json"

if not (cfg.reuse_artifacts and final_weights.exists() and final_hparams.exists() and final_history_json.exists()):
    final_vae, final_history = train_vae_final(
        X_trainval,
        out_dir=final_vae_dir,
        epochs=ksum["suggested_epochs_for_final"],
        seed=cfg.seed,
        **{k: vae_hparams[k] for k in [
            "hidden_dims", "latent_dim", "beta_target", "warmup_ep", "l2_reg", "lr",
            "recon_kind", "drop_rate", "activation", "norm_kind", "batch_size", "clipnorm",
        ]},
    )
else:
    final_vae = load_vae_from_dir(final_vae_dir)
    final_history = load_history(final_history_json)

print("Final VAE ready:", final_weights.exists())


KFold mean val recon loss: 914.4014282226562
Suggested epochs for final: 96
Final VAE ready: True


2026-02-16 17:34:26.569072: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/nico/Desktop/5to/TesisFinal/.tesis-final-venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 30 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### Curvas de entrenamiento del VAE

Se grafican las métricas de entrenamiento del VAE final: pérdida total, pérdida de reconstrucción, divergencia KL, y las métricas de validación (Pearson y coseno) cuando están disponibles.

In [9]:
fig_path = paths.out_dir / "figures" / "vae_final_history.png"
plot_vae_history(final_history, fig_path)
print("Saved:", fig_path)


Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/figures/vae_final_history.png


### Interpretación de pérdidas del VAE

Para facilitar la comprensión de los valores reportados, convertimos la pérdida de reconstrucción (que es una suma sobre todas las dimensiones de entrada) a un error promedio por elemento.

In [10]:
# Get input dimension from X_trainval shape (number of FC features)
input_dim = X_trainval.shape[1]
final_recon = final_history["recon_loss"][-1]
final_kl = final_history["kl_loss"][-1]
final_total = final_history["loss"][-1]  # Total loss (recon + beta*kl)
beta = vae_hparams["beta_target"]

# Per-feature reconstruction error
recon_per_feat = final_recon / input_dim

print("VAE Final Training Summary")
print("=" * 60)
print(f"Total loss:              {final_total:.2f}")
print(f"Reconstruction loss:     {final_recon:.2f} (sum over {input_dim} features)")
print(f"  → Per-feature MAE:     {recon_per_feat:.4f}")
print(f"KL divergence:           {final_kl:.2f}")
print(f"  → Weighted (β={beta:.4f}):  {beta * final_kl:.2f}")
print("=" * 60)
print(f"Loss decomposition: {final_recon:.2f} + {beta:.4f} × {final_kl:.2f} = {final_recon + beta*final_kl:.2f}")


VAE Final Training Summary
Total loss:              841.22
Reconstruction loss:     829.16 (sum over 6670 features)
  → Per-feature MAE:     0.1243
KL divergence:           212.84
  → Weighted (β=0.0567):  12.06
Loss decomposition: 829.16 + 0.0567 × 212.84 = 841.22


### Extracción de embeddings latentes

Se utiliza el encoder del VAE entrenado para extraer los vectores μ (media del espacio latente) de cada sujeto. Estos embeddings de 64 dimensiones reemplazan los 6670 features originales de FC y se guardan en disco como artefactos reutilizables.

In [ ]:
emb_dir = paths.out_dir / "embeddings"
emb_tr_path = emb_dir / "mu_trainval"
emb_te_path = emb_dir / "mu_test"

if not (cfg.reuse_artifacts and emb_tr_path.with_suffix(".npy").exists()):
    Z_tr = encode_mu(final_vae.encoder, X_trainval)
    save_embeddings(emb_tr_path, trainval_ids, Z_tr)
else:
    ids_tr_loaded, Z_tr = load_embeddings(emb_tr_path)
    assert ids_tr_loaded == trainval_ids, "Trainval embedding IDs do not match split IDs."

if not (cfg.reuse_artifacts and emb_te_path.with_suffix(".npy").exists()):
    Z_te = encode_mu(final_vae.encoder, X_test)
    save_embeddings(emb_te_path, test_ids, Z_te)
else:
    ids_te_loaded, Z_te = load_embeddings(emb_te_path)
    assert ids_te_loaded == test_ids, "Test embedding IDs do not match split IDs."

print("Z_tr:", Z_tr.shape, "Z_te:", Z_te.shape)


Z_tr: (1120, 64) Z_te: (125, 64)


### Reconstrucciones del VAE

Se generan visualizaciones lado a lado de las matrices FC originales vs. las reconstruidas por el decoder, para verificar cualitativamente la calidad de la compresión. Cada par incluye la correlación de Pearson entre el vector original y el reconstruido.

In [ ]:
_, _, z_test = final_vae.encoder.predict(X_test, verbose=0)
X_test_pred = final_vae.decoder.predict(z_test, verbose=0)

recon_dir = paths.out_dir / "figures" / "vae_recons"
plot_reconstructions(X_test, X_test_pred, vector_to_matrix, recon_dir, n=5)
print("Saved:", recon_dir)


Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/figures/vae_recons


## Predicción de Edad con XGBoost

Optimización de hiperparámetros, entrenamiento y evaluación del regresor final sobre features combinadas (embeddings VAE + T1w + sexo).

In [ ]:
_cdf = cohort_df.set_index("record_id") if "record_id" in cohort_df.columns else cohort_df
t1_cols = [c for c in _cdf.columns if c.startswith("t1_")]
diag_map = {"CN": 0.0, "AD": 1.0, "FTD": 2.0}

def get_y_sex_diag_t1(ids):
    """Extract target and covariate arrays aligned to a list of IDs."""
    y = _cdf.loc[ids, "age"].to_numpy(dtype=float)
    sex = (_cdf.loc[ids, "sex"].astype(str) == "Male").to_numpy(dtype=np.float32)
    diag = _cdf.loc[ids, "diagnosis"].map(diag_map).to_numpy(dtype=np.float32)
    if np.any(~np.isfinite(diag)):
        raise ValueError("Unknown diagnosis labels found.")
    T1 = _cdf.loc[ids, t1_cols].to_numpy(dtype=np.float32)
    return y, sex, diag, T1

ytr, sextr, diagtr, T1_tr = get_y_sex_diag_t1(trainval_ids)
yte, sexte, diagte, T1_te = get_y_sex_diag_t1(test_ids)

Xtr = build_feats(Z=Z_tr, T1=T1_tr, sex=sextr)
Xte = build_feats(Z=Z_te, T1=T1_te, sex=sexte)
Xtr, ytr = clean_xy(Xtr, ytr)
Xte, yte = clean_xy(Xte, yte)

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)


Xtr: (1120, 181) Xte: (125, 181)


### Optimización, entrenamiento y evaluación de XGBoost

Se optimizan los hiperparámetros de XGBoost con Optuna (100 trials, 5-fold CV), se entrena el modelo final sobre trainval completo, y se evalúa en el conjunto de test. Se generan el scatter plot de predicción y el gráfico de importancia de features.

In [ ]:
from src.optuna_xgb import tune_xgb_with_cv

optuna_out = paths.out_dir / "optuna" / "xgb_best_cv.json"
xgb_model_path = paths.out_dir / "models" / "xgb_final.json"

if not (cfg.reuse_artifacts and optuna_out.exists()):
    print(f"Running XGB Optuna: {cfg.optuna_xgb_trials} trials, {cfg.k_folds}-fold CV...")
    best_params = tune_xgb_with_cv(
        Xtr, ytr,
        seed=cfg.seed,
        n_trials=cfg.optuna_xgb_trials,
        k_folds=cfg.k_folds,
        out_path=optuna_out,
    )
else:
    with open(optuna_out, "r", encoding="utf-8") as f:
        best_params = json.load(f)["best_params"]
    print("Loaded cached XGB params:", optuna_out)

print("Best params:", best_params)

from xgboost import XGBRegressor

if cfg.reuse_artifacts and xgb_model_path.exists():
    xgb_model = XGBRegressor()
    xgb_model.load_model(str(xgb_model_path))
else:
    xgb_model = train_xgb(Xtr, ytr, best_params)
    xgb_model_path.parent.mkdir(parents=True, exist_ok=True)
    xgb_model.save_model(str(xgb_model_path))

res = eval_xgb(xgb_model, Xte, yte)
pred_te = res["pred"]

m = regression_metrics(yte, pred_te)
print("Test metrics:", m)

metrics_path = paths.out_dir / "metrics" / "xgb_test_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(m, f, indent=2)

scatter_path = paths.out_dir / "figures" / "xgb_scatter.png"
plot_pred_scatter(yte, pred_te, scatter_path,
                  title=f"Test  MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  R²={m['R2']:.2f}")

feature_names = build_feature_names(
    z_dim=Z_tr.shape[1], t1_dim=T1_tr.shape[1], use_sex=True, use_diag=False,
)
imp_path = paths.out_dir / "figures" / "xgb_importance_top20.png"
plot_feature_importance(xgb_model, feature_names, imp_path, top_k=20)

print("Saved:", metrics_path)
print("Saved:", scatter_path)
print("Saved:", imp_path)


Loaded cached XGB params: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/optuna/xgb_best_cv.json
Best params: {'n_estimators': 2103, 'max_depth': 6, 'learning_rate': 0.011042092255313062, 'subsample': 0.5021328242984182, 'colsample_bytree': 0.9505971249574193, 'reg_alpha': 0.0013554585093299317, 'reg_lambda': 7.0841939500925095, 'min_child_weight': 1.1415352944950146, 'gamma': 0.6749458207517294, 'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mae'}
Test metrics: {'MAE': 5.704533813476562, 'RMSE': 7.336762246654898, 'R2': 0.389137380561027, 'Pearson': 0.6259362876071806}
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/metrics/xgb_test_metrics.json
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/figures/xgb_scatter.png
Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/figures/xgb_importance_top20.png


## Ablaciones

Evaluación sistemática de la contribución de cada fuente de features al rendimiento predictivo. Se utilizan los mismos hiperparámetros de XGBoost optimizados por validación cruzada para garantizar una comparación justa entre configuraciones.

In [ ]:
from xgboost import XGBRegressor

def run_xgb_experiment(name, Xtr_i, ytr_i, Xte_i, yte_i, params):
    """Train and evaluate a single ablation configuration."""
    model = XGBRegressor(**params)
    model.fit(Xtr_i, ytr_i, verbose=False)
    pred = model.predict(Xte_i)
    m = regression_metrics(yte_i, pred)
    print(f"{name:40s} | MAE={m['MAE']:.2f} RMSE={m['RMSE']:.2f} R²={m['R2']:.3f} r={m['Pearson']:.3f}")
    return m


### Ejecución de las ablaciones

Se ejecutan las 7 configuraciones del estudio de ablaciones y se guardan los resultados en JSON. Cada configuración usa los mismos hiperparámetros de XGBoost para una comparación justa.

In [ ]:
ytr_abl, sextr_abl, diagtr_abl, T1_tr_abl = get_y_sex_diag_t1(trainval_ids)
yte_abl, sexte_abl, diagte_abl, T1_te_abl = get_y_sex_diag_t1(test_ids)

experiments = [
    ("T1w",                             dict(T1=T1_tr_abl),                                          dict(T1=T1_te_abl)),
    ("T1w + sex",                       dict(T1=T1_tr_abl, sex=sextr_abl),                           dict(T1=T1_te_abl, sex=sexte_abl)),
    ("VAE(Z)",                          dict(Z=Z_tr),                                                 dict(Z=Z_te)),
    ("VAE(Z) + sex",                    dict(Z=Z_tr, sex=sextr_abl),                                  dict(Z=Z_te, sex=sexte_abl)),
    ("VAE(Z) + T1w",                    dict(Z=Z_tr, T1=T1_tr_abl),                                  dict(Z=Z_te, T1=T1_te_abl)),
    ("VAE(Z) + T1w + sex",             dict(Z=Z_tr, T1=T1_tr_abl, sex=sextr_abl),                   dict(Z=Z_te, T1=T1_te_abl, sex=sexte_abl)),
    ("VAE(Z) + T1w + sex + diagnosis", dict(Z=Z_tr, T1=T1_tr_abl, sex=sextr_abl, diag=diagtr_abl), dict(Z=Z_te, T1=T1_te_abl, sex=sexte_abl, diag=diagte_abl)),
]

print(f"{'Experiment':40s} | {'MAE':>5s} {'RMSE':>5s} {'R²':>6s} {'r':>6s}")
print("-" * 70)

results = {}
for name, tr_kwargs, te_kwargs in experiments:
    Xtr_i, ytr_i = clean_xy(build_feats(**tr_kwargs), ytr_abl)
    Xte_i, yte_i = clean_xy(build_feats(**te_kwargs), yte_abl)
    results[name] = run_xgb_experiment(name, Xtr_i, ytr_i, Xte_i, yte_i, best_params)

abl_path = paths.out_dir / "metrics" / "ablation_results.json"
abl_path.parent.mkdir(parents=True, exist_ok=True)
with open(abl_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {abl_path}")


Experiment                               |   MAE  RMSE     R²      r
----------------------------------------------------------------------
T1w                                      | MAE=5.89 RMSE=7.48 R²=0.366 r=0.606
T1w + sex                                | MAE=5.88 RMSE=7.44 R²=0.372 r=0.612
VAE(Z)                                   | MAE=7.01 RMSE=9.12 R²=0.056 r=0.284
VAE(Z) + sex                             | MAE=7.06 RMSE=9.18 R²=0.044 r=0.268
VAE(Z) + T1w                             | MAE=5.62 RMSE=7.20 R²=0.411 r=0.644
VAE(Z) + T1w + sex                       | MAE=5.70 RMSE=7.34 R²=0.389 r=0.626
VAE(Z) + T1w + sex + diagnosis           | MAE=5.64 RMSE=7.17 R²=0.416 r=0.648

Saved: /home/nico/Desktop/5to/TesisFinal/Thesis-comp-sci/Outputs/metrics/ablation_results.json
